# BTC chord detection + melody compatibility

This notebook runs the published BTC checkpoint directly. Set `AUDIO_PATH` to analyze your own recording; the bundled BTC example is used by default.

In [1]:
import os
from pathlib import Path

import matplotlib.pyplot as plt

from reharmonize_models import (
    aggregate_melody_by_chord, detect_chords, melody_clash_score,
    melody_pitch_track, save_lab,
)

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
AUDIO = Path(os.environ.get('AUDIO_PATH', ROOT / 'vendor/BTC-ISMIR19/test/example.mp3'))
DURATION = float(os.environ.get('ANALYSIS_DURATION', '30'))
AUDIO, DURATION

SyntaxError: invalid syntax (1810483327.py, line 14)

In [ ]:
chords = detect_chords(AUDIO, btc_root=ROOT / 'vendor/BTC-ISMIR19', duration=DURATION)
lab_path = save_lab(chords, ROOT / 'artifacts' / f'{AUDIO.stem}.lab')
print(f'Saved {len(chords)} chord intervals to {lab_path}')
chords

In [ ]:
melody = melody_pitch_track(AUDIO, duration=DURATION)
segments = aggregate_melody_by_chord(chords, melody)
segments['clash'] = segments.apply(
    lambda row: melody_clash_score(row.melody_profile, row.chord), axis=1
)
segments[['start', 'end', 'chord', 'voiced_frames', 'clash']]

In [ ]:
ax = melody.plot(x='time', y='midi', figsize=(14, 4), legend=False, title='BTC chords over CREPE melody')
for row in chords.itertuples(index=False):
    ax.axvline(row.start, color='gray', alpha=0.2)
    ax.text((row.start + row.end) / 2, ax.get_ylim()[1], row.chord, ha='center', va='top', fontsize=8)
ax.set(xlabel='Time (seconds)', ylabel='MIDI pitch')
plt.show()

The chord labels now come from BTC rather than fallback data. The `clash` value is a diagnostic feature for candidate ranking; it is not yet a trained recommendation score.